In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
delivery_time_counts = df['Delivery_Time'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(delivery_time_counts.index, delivery_time_counts.values, color='coral')
plt.title('Dilivery Time Distribution')
plt.xlabel('Dilivery Time')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Task 1: Write your code here:
df.drop( columns= ['Order_ID'])

In [ ]:
# Task 2: Write your code here:
#Checking if there is missing values
def check_missing_values(df):
 # Get missing values using pandas
   missing_values = df.isnull().sum()
   print("Missing Values per Column:")
   print(missing_values[missing_values > 0])
   if missing_values.any():
       print("\nHandle Missing Values as needed.")
   else:
       print("\nNo Missing Values Found.")
check_missing_values(df)

#Handle the missing values
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())

df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])

print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
#Check for duplicates and remove them
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
       print("Dropping Duplicates...")
       df.drop_duplicates(inplace=True)
       print("Duplicates Dropped.")
    else:
      print("No Duplicate Samples Found.")
check_duplicates(df)


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

#Label Encoder
categorical_cols = ['Weather', 'Time_of_Day', 'Vehicle_Type', 'Traffic_Level']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
df.head()


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler
print('data before scaling:\n', df) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(df) # Apply fi
print('\nData after scaling:\n', data_standard_scaled) #show after scaling


In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    df['Delivery_Time'].hist()
    plt.show()
check_target_imbalance(df, "delivery_time")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
feature_cols = ['Order_ID',	'Distance_km',	'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs']
X = df[feature_cols]
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

# Predict and evaluate
y_pred = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: ${mae:,.2f}")

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []
rmse_scores = []
for train_idx, val_idx in kfold.split(X_train_scaled):
   X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
   y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
# Train and predict
model.fit(X_fold_train, y_fold_train)
y_fold_pred = model.predict(X_fold_val)
# Calculate metrics
mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))
mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)
print(f"5-Fold CV Results:")
print(f"MAE: ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({'feature': feature_cols,'importance': model.feature_importances_}).sort_values('importance', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()


In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: